In [2]:
from pathlib import Path
from typing import List, Literal, get_args
from getpass import getpass

import instructor
import pandas as pd

from pydantic import BaseModel, Field
from tqdm import tqdm

In [3]:
OPENAI_API_KEY = getpass("Enter your OpenAI API key: ")

if not OPENAI_API_KEY:
    raise RuntimeError("OpenAI API key was not provided.")

MODEL_ID = "gpt-4.1-mini"
SAMPLE_SIZE = 10
RANDOM_STATE = 42

In [4]:
client = instructor.from_provider(
    f"openai/{MODEL_ID}",
    api_key=OPENAI_API_KEY,
)

print(f"Model: {MODEL_ID}")

Model: gpt-4.1-mini


In [5]:
def call_api(
    messages,
    response_model,
    max_retries=2,
):
    response = client.create(
        messages=messages,
        response_model=response_model,
        temperature=0,
        max_retries=max_retries,
    )

    return response

In [6]:
data_files = [
    Path("jobs.parquet"),
    Path("skills_bielik.parquet"),
    Path("skills_gemini.parquet"),
]

for file_path in data_files:
    print(f"{file_path.name}: {file_path.exists()}")

jobs.parquet: True
skills_bielik.parquet: True
skills_gemini.parquet: True


In [7]:
df_jobs = pd.read_parquet("jobs.parquet")

print(df_jobs.shape)
print(df_jobs.columns.tolist())

df_jobs.sample(5, random_state=RANDOM_STATE)

(3946, 7)
['raw_title', 'raw_location', 'company_name', 'salary_usd_min', 'salary_usd_max', 'level', 'raw_description']


,raw_title,raw_location,company_name,salary_usd_min,salary_usd_max,level,raw_description
2659,Senior Machine Learning Engineer,Tel Aviv,HUMAN,131.0,201.0,Senior-level / Expert,View company page\n\nView company page\n\nAppl...
1110,Senior Data Engineer,"Heredia, Costa Rica",Experian,115.0,180.0,Senior-level / Expert,View company page\n\nView company page\n\nAppl...
2440,"Senior Data Engineer, US",US - San Francisco,Airwallex,121.0,190.0,Senior-level / Expert,View company page\n\nView company page\n\nAppl...
70,Sr. Machine Learning Engineer,"Seoul, South Korea",Match Group,149.0,230.0,Senior-level / Expert,View company page\n\nView company page\n\nAppl...
1517,(Global) Research Engineer· Lunit INSIGHT,"Seoul, Seoul, Korea, Republic of",Lunit,142.0,211.0,Undefined,View company page\n\nView company page\n\nAppl...


In [8]:
sample_job = df_jobs.sample(
    1,
    random_state=RANDOM_STATE,
).iloc[0]

sample_title = sample_job["raw_title"]
sample_description = sample_job["raw_description"]

print(sample_title)
print()
print(sample_description)

Senior Machine Learning Engineer

View company page

View company page

Apply now

Apply later
Posted 3 weeks ago

### What you'll do:
- Design and develop MLOps infrastructure to support our ML/AI models at scale, including CI/CD, automation, evaluation, and monitoring.
- Build backend infrastructure (data processing jobs, micro-services), create automated workflows to process large datasets for machine learning purposes.
- Work closely with diverse teams such as Data Science, Product Management, Engineering and DevOps.


### Who you are:
- At least 5+ years experience in backend/data engineering writing code that deployed and used in production.
- 2+ years of experience as a Machine Learning EngineerProficient in Python and its ecosystem.
- Experience designing and operating large ML inference platforms.
- Experience with one or more of these technologies: Vertex AI, Kubeflow, Spark, Kafka, SageMaker, Databricks.
- Familiarity with various ML algorithms and frameworks (PyTorch, Tenso

In [9]:
class JobPostingHarvest(BaseModel):
    skills: List[str] = Field(
        description=(
            "Names of technologies, programming languages, frameworks, "
            "platforms, databases, tools, protocols, and technical concepts "
            "required or mentioned in the job posting."
        )
    )

In [10]:
skill_extraction_prompt = """
Extract technical skills from the provided job description.

Include:
- programming languages
- frameworks
- libraries
- cloud platforms
- databases
- data engineering tools
- machine learning tools
- DevOps tools
- technical methodologies and concepts

Use concise and commonly recognized names.

Do not include:
- soft skills
- company names
- job titles
- generic responsibilities
- benefits
"""

In [11]:
result = call_api(
    messages=[
        {
            "role": "system",
            "content": skill_extraction_prompt,
        },
        {
            "role": "user",
            "content": sample_description,
        },
    ],
    response_model=JobPostingHarvest,
)

result

JobPostingHarvest(skills=['Python', 'PyTorch', 'TensorFlow', 'HuggingFace', 'Scikit-learn', 'R', 'Golang', 'Spark', 'Kafka', 'MongoDB', 'Snowflake', 'BigQuery', 'Redshift', 'Databricks', 'SageMaker', 'Vertex AI', 'Kubeflow', 'Airflow', 'Docker', 'CI/CD', 'MLOps', 'DevOps', 'Cloud platforms (GCP)', 'Machine Learning', 'ML inference platforms', 'Data processing', 'Micro-services', 'Automated workflows', 'Production ML system monitoring'])

In [12]:
result.skills

['Python',
 'PyTorch',
 'TensorFlow',
 'HuggingFace',
 'Scikit-learn',
 'R',
 'Golang',
 'Spark',
 'Kafka',
 'MongoDB',
 'Snowflake',
 'BigQuery',
 'Redshift',
 'Databricks',
 'SageMaker',
 'Vertex AI',
 'Kubeflow',
 'Airflow',
 'Docker',
 'CI/CD',
 'MLOps',
 'DevOps',
 'Cloud platforms (GCP)',
 'Machine Learning',
 'ML inference platforms',
 'Data processing',
 'Micro-services',
 'Automated workflows',
 'Production ML system monitoring']

In [13]:
samples = df_jobs.sample(
    SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)["raw_description"].tolist()

len(samples)

10

In [14]:
sample_skills = []

for sample_description in tqdm(samples):
    result = call_api(
        messages=[
            {
                "role": "system",
                "content": skill_extraction_prompt,
            },
            {
                "role": "user",
                "content": sample_description,
            },
        ],
        response_model=JobPostingHarvest,
    )

    sample_skills.extend(result.skills)

100%|██████████| 10/10 [00:22<00:00,  2.29s/it]


In [15]:
print(f"Extracted skills: {len(sample_skills)}")
print(f"Unique skills: {len(set(sample_skills))}")

Extracted skills: 193
Unique skills: 152


In [16]:
sample_skill_counts = (
    pd.Series(sample_skills)
    .value_counts()
    .rename_axis("skill")
    .reset_index(name="count")
)

sample_skill_counts.head(30)

,skill,count
0,Python,8
1,Machine Learning,5
2,PyTorch,4
3,SQL,4
4,Spark,3
5,MLOps,3
6,BigQuery,3
7,TensorFlow,3
8,R,3
9,Snowflake,2


In [17]:
df_skills_bielik = pd.read_parquet(
    "skills_bielik.parquet"
)

print(df_skills_bielik.shape)

df_skills_bielik.head(20)

(6754, 2)


,skill,count
0,Python,2496
1,SQL,1948
2,Machine Learning,1550
3,AWS,970
4,Statistics,909
5,Agile,791
6,Spark,710
7,R,701
8,Tableau,700
9,ETL,648


In [18]:
df_skills_gemini = pd.read_parquet(
    "skills_gemini.parquet"
)

print(df_skills_gemini.shape)

df_skills_gemini.head(20)

(7601, 2)


,skill,count
0,Python,2549
1,SQL,2050
2,Machine Learning,1474
3,AWS,1066
4,Agile,809
5,Spark,791
6,R,787
7,ETL,763
8,Statistics,734
9,Tableau,732


In [19]:
print(df_skills_bielik.columns.tolist())
print(df_skills_gemini.columns.tolist())

['skill', 'count']
['skill', 'count']


In [20]:
df_both = pd.merge(
    df_skills_bielik,
    df_skills_gemini,
    on="skill",
    how="outer",
)

df_both.columns = [
    "skill",
    "count_bielik",
    "count_gemini",
]

df_both[["count_bielik", "count_gemini"]] = (
    df_both[["count_bielik", "count_gemini"]]
    .fillna(0)
    .astype(int)
)

df_both.head()

,skill,count_bielik,count_gemini
0,,0,1
1,Research,1,0
2,*nix systems,0,2
3,.NET,8,18
4,.NET Core,0,2


In [21]:
df_both = df_both.sort_values(
    by="count_gemini",
    ascending=False,
)

df_both.head(50)

,skill,count_bielik,count_gemini
7978,Python,2496,2549
8732,SQL,1948,2050
6109,Machine Learning,1550,1474
281,AWS,970,1066
531,Agile,791,809
9356,Spark,710,791
8115,R,701,787
3722,ETL,648,763
9536,Statistics,909,734
9806,Tableau,700,732


In [22]:
df_both["count_diff"] = (
    df_both["count_bielik"]
    - df_both["count_gemini"]
)

df_both["absolute_diff"] = (
    df_both["count_diff"].abs()
)

In [23]:
df_both.sort_values(
    by="absolute_diff",
    ascending=False,
).head(20)

,skill,count_bielik,count_gemini,count_diff,absolute_diff
3887,Engineering,607,111,496,496
8469,Research,420,105,315,315
2218,Computer Science,518,259,259,259
6281,Mathematics,363,133,230,230
100,AI,190,410,-220,220
3087,Data pipelines,371,162,209,209
2852,Data Pipelines,183,380,-197,197
9976,Testing,539,345,194,194
4223,Finance,207,21,186,186
8985,Security,459,277,182,182


In [24]:
top_skills = (
    df_both
    .sort_values(
        by="count_gemini",
        ascending=False,
    )
    .head(100)
)

top_skills

,skill,count_bielik,count_gemini,count_diff,absolute_diff
7978,Python,2496,2549,-53,53
8732,SQL,1948,2050,-102,102
6109,Machine Learning,1550,1474,76,76
281,AWS,970,1066,-96,96
531,Agile,791,809,-18,18
...,...,...,...,...,...
6418,Microservices,84,110,-26,26
3531,Distributed Systems,102,110,-8,8
2788,Data Lake,17,109,-92,92
3895,English,31,108,-77,77


In [25]:
SkillName = Literal[
    "Python",
    "R",
    "Java",
    "Scala",
    "C#",
    "C++",
    "JavaScript",
    "TypeScript",
    "SQL",
    "Go",

    "PyTorch",
    "TensorFlow",
    "scikit-learn",
    "Keras",
    "XGBoost",
    "LightGBM",
    "Hugging Face Transformers",
    "OpenAI API",
    "LangChain",
    "OpenCV",

    "MLflow",
    "Kubeflow",
    "Weights & Biases",
    "SageMaker",
    "Vertex AI",
    "Azure ML",

    "AWS",
    "Azure",
    "GCP",

    "PostgreSQL",
    "MySQL",
    "SQL Server",
    "Oracle",
    "MongoDB",
    "Cassandra",
    "Redis",
    "Neo4j",
    "Elasticsearch",
    "DynamoDB",

    "Snowflake",
    "Redshift",
    "BigQuery",
    "Azure Synapse",
    "Databricks",
    "Vertica",

    "Apache Spark",
    "PySpark",
    "Hadoop",
    "Apache Kafka",
    "Apache Flink",
    "Hive",
    "Presto/Trino",
    "Delta Lake",

    "Apache Airflow",
    "DBT",
    "Talend",
    "Informatica",
    "SSIS",
    "Fivetran",
    "AWS Glue",
    "Azure Data Factory",

    "Tableau",
    "Power BI",
    "Looker",
    "Grafana",
    "Plotly",
    "QlikView/Qlik Sense",

    "Docker",
    "Kubernetes",
    "Terraform",
    "Ansible",
    "Jenkins",
    "GitLab CI",
    "GitHub Actions",
    "Helm",
    "Prometheus",
    "Git",

    "React",
    "Angular",
    "Vue.js",
    "Spring Boot",
    ".NET",
    "Django",
    "Flask",
    "FastAPI",
    "Node.js",

    "RabbitMQ",
    "Amazon Kinesis",
    "Amazon SQS",
    "Google Pub/Sub",

    "Machine Learning",
    "Deep Learning",
    "Natural Language Processing",
    "Computer Vision",
    "Generative AI",
    "Large Language Models",
    "RAG",
    "Embeddings",
    "Feature Engineering",
]

In [26]:
SkillCategory = Literal[
    "programming_language",
    "ml_ai_library",
    "mlops_platform",
    "cloud_platform",
    "database",
    "data_warehouse",
    "big_data_processing",
    "data_engineering_tool",
    "bi_visualization",
    "devops_infra",
    "web_framework",
    "messaging_streaming",
    "ml_ai_concept",
]

In [27]:
allowed_skills = get_args(SkillName)
allowed_categories = get_args(SkillCategory)

print(f"Allowed skills: {len(allowed_skills)}")
print(f"Allowed categories: {len(allowed_categories)}")

Allowed skills: 99
Allowed categories: 13


In [28]:
allowed_skill_set = set(
    get_args(SkillName)
)

top_100_skill_set = set(
    df_both
    .sort_values(
        by="count_gemini",
        ascending=False,
    )
    .head(100)["skill"]
)

covered_skills = (
    top_100_skill_set
    & allowed_skill_set
)

missing_skills = (
    top_100_skill_set
    - allowed_skill_set
)

coverage = (
    len(covered_skills)
    / len(top_100_skill_set)
)

print(f"Coverage: {coverage:.1%}")
print(f"Missing skills: {len(missing_skills)}")

Coverage: 35.0%
Missing skills: 65


In [29]:
sorted(missing_skills)

['AI',
 'API',
 'APIs',
 'Agile',
 'Airflow',
 'Algorithms',
 'Architecture',
 'Big Data',
 'Business Intelligence',
 'CI/CD',
 'Classification',
 'Cloud',
 'Clustering',
 'Computer Science',
 'Data Analysis',
 'Data Analytics',
 'Data Engineering',
 'Data Governance',
 'Data Lake',
 'Data Management',
 'Data Mining',
 'Data Modeling',
 'Data Pipelines',
 'Data Quality',
 'Data Science',
 'Data Visualization',
 'Data Warehouse',
 'Data Warehousing',
 'Data analysis',
 'Data pipelines',
 'Data visualization',
 'DataOps',
 'DevOps',
 'Distributed Systems',
 'ELT',
 'ETL',
 'Engineering',
 'English',
 'Excel',
 'GitHub',
 'Google Cloud',
 'Kafka',
 'LLMs',
 'Linux',
 'ML',
 'ML Models',
 'MLOps',
 'Mathematics',
 'Microservices',
 'NLP',
 'NoSQL',
 'NumPy',
 'Pandas',
 'Pipelines',
 'RDBMS',
 'S3',
 'SAS',
 'Scikit-learn',
 'Scrum',
 'Security',
 'Spark',
 'Statistics',
 'Streaming',
 'Testing',
 'dbt']

In [30]:
class SkillRequirement(BaseModel):
    name: SkillName = Field(
        description=(
            "Canonical skill name selected only "
            "from the allowed SkillName taxonomy."
        )
    )

    category: SkillCategory = Field(
        description=(
            "Canonical category describing "
            "the type of technical skill."
        )
    )

    required: bool = Field(
        description=(
            "Whether the skill is explicitly required "
            "for the job rather than merely preferred."
        )
    )

In [31]:
class JobSkillsProfile(BaseModel):
    skills: List[SkillRequirement] = Field(
        description=(
            "Technical skills identified in the job description "
            "and normalized using the predefined taxonomy."
        )
    )

In [32]:
taxonomy_prompt = """
Analyze the provided job description and extract technical skills.

Follow these rules:

1. Use only skill names allowed by the response schema.
2. Never invent a new skill name.
3. Normalize equivalent terminology to the closest canonical skill.
4. Assign exactly one category to every extracted skill.
5. Mark a skill as required only when the job description clearly
   presents it as a requirement.
6. Ignore soft skills, company names, benefits, and generic responsibilities.
7. If a mentioned technology is not available in the taxonomy,
   do not invent a replacement unless there is a clear canonical equivalent.
"""

In [33]:
sample_job = df_jobs.sample(
    1,
    random_state=RANDOM_STATE,
).iloc[0]

sample_title = sample_job["raw_title"]
sample_description = sample_job["raw_description"]

print(sample_title)

Senior Machine Learning Engineer


In [34]:
job_profile = call_api(
    messages=[
        {
            "role": "system",
            "content": taxonomy_prompt,
        },
        {
            "role": "user",
            "content": sample_description,
        },
    ],
    response_model=JobSkillsProfile,
)

job_profile

JobSkillsProfile(skills=[SkillRequirement(name='Python', category='programming_language', required=True), SkillRequirement(name='Vertex AI', category='mlops_platform', required=True), SkillRequirement(name='Kubeflow', category='mlops_platform', required=True), SkillRequirement(name='Apache Spark', category='big_data_processing', required=True), SkillRequirement(name='Apache Kafka', category='messaging_streaming', required=True), SkillRequirement(name='SageMaker', category='mlops_platform', required=True), SkillRequirement(name='Databricks', category='data_engineering_tool', required=True), SkillRequirement(name='PyTorch', category='ml_ai_library', required=True), SkillRequirement(name='TensorFlow', category='ml_ai_library', required=True), SkillRequirement(name='Hugging Face Transformers', category='ml_ai_library', required=True), SkillRequirement(name='scikit-learn', category='ml_ai_library', required=True), SkillRequirement(name='Snowflake', category='data_warehouse', required=True),

In [35]:
print(
    job_profile.model_dump_json(
        indent=2
    )
)

{
  "skills": [
    {
      "name": "Python",
      "category": "programming_language",
      "required": true
    },
    {
      "name": "Vertex AI",
      "category": "mlops_platform",
      "required": true
    },
    {
      "name": "Kubeflow",
      "category": "mlops_platform",
      "required": true
    },
    {
      "name": "Apache Spark",
      "category": "big_data_processing",
      "required": true
    },
    {
      "name": "Apache Kafka",
      "category": "messaging_streaming",
      "required": true
    },
    {
      "name": "SageMaker",
      "category": "mlops_platform",
      "required": true
    },
    {
      "name": "Databricks",
      "category": "data_engineering_tool",
      "required": true
    },
    {
      "name": "PyTorch",
      "category": "ml_ai_library",
      "required": true
    },
    {
      "name": "TensorFlow",
      "category": "ml_ai_library",
      "required": true
    },
    {
      "name": "Hugging Face Transformers",
      "category": "ml

In [36]:
skills_df = pd.DataFrame(
    [
        skill.model_dump()
        for skill in job_profile.skills
    ]
)

skills_df

,name,category,required
0,Python,programming_language,True
1,Vertex AI,mlops_platform,True
2,Kubeflow,mlops_platform,True
3,Apache Spark,big_data_processing,True
4,Apache Kafka,messaging_streaming,True
5,SageMaker,mlops_platform,True
6,Databricks,data_engineering_tool,True
7,PyTorch,ml_ai_library,True
8,TensorFlow,ml_ai_library,True
9,Hugging Face Transformers,ml_ai_library,True


In [37]:
cloud_skills = skills_df[
    skills_df["category"]
    == "cloud_platform"
]

cloud_skills

,name,category,required
